In [1]:
import json
from pathlib import Path

root = Path.cwd()
print('cwd', root)

results_baseline = json.loads((Path('evaluation') / 'results_baseline.json').read_text(encoding='utf-8'))
results_improved = json.loads((Path('evaluation') / 'results_improved.json').read_text(encoding='utf-8'))

print('baseline p@5', results_baseline['p_at_5'])
print('improved best p@5', results_improved['best']['p_at_5'])
print('improved config', results_improved['best']['config'])

from pprint import pprint
pprint(results_improved['best'])


cwd c:\Users\vansh\OneDrive\Desktop\PrepPanda\evaluation


FileNotFoundError: [Errno 2] No such file or directory: 'evaluation\\results_baseline.json'

In [2]:
from pathlib import Path

root = Path.cwd().parent
print('root', root)
results_baseline = json.loads((root / 'evaluation' / 'results_baseline.json').read_text(encoding='utf-8'))
results_improved = json.loads((root / 'evaluation' / 'results_improved.json').read_text(encoding='utf-8'))
print('baseline p@5', results_baseline['p_at_5'])
print('improved best p@5', results_improved['best']['p_at_5'])
print('improved config', results_improved['best']['config'])


root c:\Users\vansh\OneDrive\Desktop\PrepPanda
baseline p@5 0.0
improved best p@5 0.04000000000000001
improved config {'semantic_weight': 0.5, 'keyword_weight': 0.5, 'rerank': True, 'context_filter': True}


In [3]:
dataset = json.loads((root / 'evaluation' / 'dataset_labeled.json').read_text(encoding='utf-8'))
print('dataset chapter_id', dataset.get('chapter_id'))
print('questions', len(dataset['questions']))
for idx, item in enumerate(dataset['questions'][:10], 1):
    print(f'--- q{idx}:', item['question'])
    print('chapter_id', item['chapter_id'])
    print('labels', item['graded_relevance'])
    for chunk in item['chunks']:
        print('  chunk', chunk['id'], repr(chunk['text'][:80]))
    print()


dataset chapter_id fba40315-57e4-4ca9-b72b-e69b1597d32d
questions 15
--- q1: What is spermatogenesis?
chapter_id e9b3f002-f293-40fe-8591-87f6dc44da5d
labels [0, 0, 0, 0, 0]
  chunk 3bfbddab-18b9-4988-95d6-c2ec4362e6be 'India was amongst the first countries in the world to\ninitiate action plans and '
  chunk a9c9d5a6-4e8b-4857-8a47-394f64757c00 'Problems and Strategies'
  chunk 8213075e-6b01-4659-8812-4385af27cdfd 'Population Explosion and'
  chunk b50fd678-75c5-4d87-805e-c1355b2faeed 'reproduction-related areas are currently in operation under the\npopular name ‘Re'
  chunk 93b51694-ddb0-416d-bb9d-d7800dddabc9 'by scientists at Central Drug Research Institute (CDRI) in Lucknow, India?\nBette'

--- q2: Describe structure of testis
chapter_id e9b3f002-f293-40fe-8591-87f6dc44da5d
labels [0, 0, 0, 0, 0]
  chunk e0aa3f7b-f09f-4ce9-98e2-e1b94fb66502 'You have learnt about human reproductive system and its\nfunctions in Chapter 2. '
  chunk 3bfbddab-18b9-4988-95d6-c2ec4362e6be 'India was amo

In [4]:
from collections import Counter

for idx, item in enumerate(dataset['questions'][:10], 1):
    print(f'q{idx}: {item["question"]}')
    print('  chapter_id:', item['chapter_id'])
    print('  label counts:', Counter(item['graded_relevance']))
    print('  first chunk prefix:', item['chunks'][0]['text'][:120].replace('\n',' '))
    print()


q1: What is spermatogenesis?
  chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  label counts: Counter({0: 5})
  first chunk prefix: India was amongst the first countries in the world to initiate action plans and programmes at a national level to attain

q2: Describe structure of testis
  chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  label counts: Counter({0: 5})
  first chunk prefix: You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a closely related top

q3: What is fertilisation in humans?
  chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  label counts: Counter({0: 5})
  first chunk prefix: You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a closely related top

q4: Explain implantation process
  chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  label counts: Counter({0: 5})
  first chunk prefix: India was amongst the first countries in the world to initiate action plans and pr

In [5]:
for idx, item in enumerate(dataset['questions'][:5], 1):
    print(f'q{idx}: {item["question"]}')
    print('  chapter_id:', item['chapter_id'])
    print('  label counts:', Counter(item['graded_relevance']))
    print('  kept chunks text sample:')
    for j, chunk in enumerate(item['chunks'][:2], 1):
        print(f'    {j}:', chunk['text'][:100].replace('\n',' '))
    print()


q1: What is spermatogenesis?
  chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  label counts: Counter({0: 5})
  kept chunks text sample:
    1: India was amongst the first countries in the world to initiate action plans and programmes at a nati
    2: Problems and Strategies

q2: Describe structure of testis
  chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  label counts: Counter({0: 5})
  kept chunks text sample:
    1: You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a
    2: India was amongst the first countries in the world to initiate action plans and programmes at a nati

q3: What is fertilisation in humans?
  chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  label counts: Counter({0: 5})
  kept chunks text sample:
    1: You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a
    2: India was amongst the first countries in the world to initiate action plans and programmes at a nat

In [8]:
import os
import asyncio
import nest_asyncio
import asyncpg

nest_asyncio.apply()

async def inspect_chapter(chapter_id, limit=15):
    dsn = (
        os.environ.get('DATABASE_URL') or
        f"postgresql://{os.environ.get('POSTGRES_USER','preppanda')}:{os.environ.get('POSTGRES_PASSWORD','preppass')}@"
        f"{os.environ.get('POSTGRES_HOST','localhost')}:{os.environ.get('POSTGRES_PORT','5432')}/"
        f"{os.environ.get('POSTGRES_DB','appdb')}"
    )
    conn = await asyncpg.connect(dsn=dsn)
    try:
        rows = await conn.fetch(
            '''
            SELECT chunk_id, position_index, content
            FROM core.chunks
            WHERE chapter_id = $1
            ORDER BY position_index
            LIMIT $2
            ''',
            chapter_id,
            limit,
        )
        print('chapter', chapter_id, 'rows', len(rows))
        for r in rows:
            print(r['position_index'], repr(r['content'][:180].replace('\n', ' ')))
    finally:
        await conn.close()

await inspect_chapter('e9b3f002-f293-40fe-8591-87f6dc44da5d', limit=15)


chapter e9b3f002-f293-40fe-8591-87f6dc44da5d rows 15
0 'You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a closely related topic – reproductive health. What do we understand by this term'
1 'India was amongst the first countries in the world to initiate action plans and programmes at a national level to attain total reproductive health as a social goal. These programme'
2 'Problems and Strategies'
3 'Population Explosion and'
4 'Medical Termination of'
5 'reproduction-related areas are currently in operation under the popular name ‘Reproductive and Child Health Care (RCH) programmes’. Creating awareness among people about various re'
6 'by scientists at Central Drug Research Institute (CDRI) in Lucknow, India? Better awareness about sex related matters, increased number of medically assisted deliveries and better '
7 'In the last century an all-round development in various fields significantly improved the quality of life of the people.

In [9]:
import os
import asyncio
import nest_asyncio
import asyncpg

nest_asyncio.apply()

dsn = (
    os.environ.get('DATABASE_URL') or
    f"postgresql://{os.environ.get('POSTGRES_USER','preppanda')}:{os.environ.get('POSTGRES_PASSWORD','preppass')}@"
    f"{os.environ.get('POSTGRES_HOST','localhost')}:{os.environ.get('POSTGRES_PORT','5432')}/"
    f"{os.environ.get('POSTGRES_DB','appdb')}"
)

async def sample_chapter(chapter_id, limit=5):
    conn = await asyncpg.connect(dsn=dsn)
    try:
        rows = await conn.fetch(
            '''SELECT position_index, content FROM core.chunks WHERE chapter_id = $1 ORDER BY position_index LIMIT $2''',
            chapter_id,
            limit,
        )
        print('chapter', chapter_id, 'rows', len(rows))
        for r in rows:
            print(r['position_index'], r['content'][:120].replace('\n', ' '))
    finally:
        await conn.close()

await sample_chapter('e9b3f002-f293-40fe-8591-87f6dc44da5d', limit=5)


chapter e9b3f002-f293-40fe-8591-87f6dc44da5d rows 5
0 You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a closely related top
1 India was amongst the first countries in the world to initiate action plans and programmes at a national level to attain
2 Problems and Strategies
3 Population Explosion and
4 Medical Termination of


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-mpnet-base-v2')
q = dataset['questions'][0]['question']
for j, chunk in enumerate(dataset['questions'][0]['chunks'], 1):
    emb_q = model.encode([q], normalize_embeddings=True)
    emb_chunk = model.encode([chunk['text']], normalize_embeddings=True)
    sim = float(np.dot(emb_q[0], emb_chunk[0]))
    print(j, 'score', sim, 'text', chunk['text'][:120].replace('\n',' '))


C:\Users\vansh\AppData\Local\Programs\Python\Python311\Lib\typing.py:1318: RuntimeWarning: coroutine 'inspect_chapter' was never awaited
  raise AttributeError(attr)
c:\Users\vansh\OneDrive\Desktop\PrepPanda\Server\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8889.74it/s]


1 score 0.240436390042305 text India was amongst the first countries in the world to initiate action plans and programmes at a national level to attain
2 score 0.09341643005609512 text Problems and Strategies
3 score 0.22092823684215546 text Population Explosion and
4 score 0.3337825536727905 text reproduction-related areas are currently in operation under the popular name ‘Reproductive and Child Health Care (RCH) p
5 score 0.287306010723114 text by scientists at Central Drug Research Institute (CDRI) in Lucknow, India? Better awareness about sex related matters, i


In [13]:
import os
import sys
from pathlib import Path
import asyncio

ROOT = Path.cwd().parent
SERVER = ROOT / 'Server'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SERVER))

from Core.Storage.PostgresHandler import PostgresHandler
from Core.Parser.embedder import ChunkEmbedder
from Core.SRS.retriever import Retriever
from evaluation.retrieval_wrapper import ImprovedRetriever

os.environ.setdefault('POSTGRES_USER', 'preppanda')
os.environ.setdefault('POSTGRES_PASSWORD', 'preppass')
os.environ.setdefault('POSTGRES_DB', 'appdb')
os.environ.setdefault('POSTGRES_HOST', 'localhost')
os.environ.setdefault('POSTGRES_PORT', '5432')

async def run_inspect():
    pg = PostgresHandler()
    await pg.connect()
    try:
        embedder = ChunkEmbedder()
        baseline = Retriever(pg=pg, embedder=embedder)
        improved = ImprovedRetriever(pg=pg, embedder=embedder)

        question = 'What is spermatogenesis?'
        chapter_id = 'e9b3f002-f293-40fe-8591-87f6dc44da5d'
        print('=== Baseline ===')
        result = await baseline.retrieve(question, chapter_id)
        for i, chunk in enumerate(result.chunks[:10], 1):
            print(i, chunk['chunk_id'], chunk['position_index'], chunk['content'][:120].replace('\n',' '))
        print('=== Improved ===')
        result2 = await improved.retrieve(question, chapter_id, semantic_weight=0.5, keyword_weight=0.5, rerank=True, use_context_filter=True, top_k=10, select_k=5)
        for i, chunk in enumerate(result2.chunks[:10], 1):
            print(i, chunk['chunk_id'], chunk.get('position_index'), chunk['content'][:120].replace('\n',' '))
    finally:
        await pg.disconnect()

import nest_asyncio
nest_asyncio.apply()
await run_inspect()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8438.90it/s]
c:\Users\vansh\OneDrive\Desktop\PrepPanda\Server\Core\Parser\embedder.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._model.get_sentence_embedding_dimension()


=== Baseline ===
1 3bfbddab-18b9-4988-95d6-c2ec4362e6be 1 India was amongst the first countries in the world to initiate action plans and programmes at a national level to attain
2 a9c9d5a6-4e8b-4857-8a47-394f64757c00 2 Problems and Strategies
3 8213075e-6b01-4659-8812-4385af27cdfd 3 Population Explosion and
4 b50fd678-75c5-4d87-805e-c1355b2faeed 5 reproduction-related areas are currently in operation under the popular name ‘Reproductive and Child Health Care (RCH) p
5 93b51694-ddb0-416d-bb9d-d7800dddabc9 6 by scientists at Central Drug Research Institute (CDRI) in Lucknow, India? Better awareness about sex related matters, i
6 ad7dcb55-8e28-4ebd-a6b1-7fb7ad6f2d6a 7 In the last century an all-round development in various fields significantly improved the quality of life of the people.
7 e239ff6d-b20d-4be8-bdd1-063bc44d8771 9 rate (MMR) and infant mortality rate (IMR) as well as an increase in number of people in reproducible age are probable r
8 65821d86-9003-4c09-b607-cad714fc7c7b 10 

In [14]:
import os
import sys
from pathlib import Path
import asyncio

ROOT = Path.cwd().parent
SERVER = ROOT / 'Server'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SERVER))

from Core.Storage.PostgresHandler import PostgresHandler
from Core.Parser.embedder import ChunkEmbedder
from Core.SRS.retriever import Retriever
from evaluation.retrieval_wrapper import ImprovedRetriever

os.environ.setdefault('POSTGRES_USER', 'preppanda')
os.environ.setdefault('POSTGRES_PASSWORD', 'preppass')
os.environ.setdefault('POSTGRES_DB', 'appdb')
os.environ.setdefault('POSTGRES_HOST', 'localhost')
os.environ.setdefault('POSTGRES_PORT', '5432')

async def run_inspect():
    pg = PostgresHandler()
    await pg.connect()
    try:
        embedder = ChunkEmbedder()
        baseline = Retriever(pg=pg, embedder=embedder)
        improved = ImprovedRetriever(pg=pg, embedder=embedder)

        question = 'What is spermatogenesis?'
        chapter_id = 'e9b3f002-f293-40fe-8591-87f6dc44da5d'
        print('=== Baseline ===')
        result = await baseline.retrieve(question, chapter_id)
        for i, chunk in enumerate(result.chunks[:3], 1):
            print(i, chunk['position_index'], chunk['content'][:80].replace('\n',' '))
        print('---')
        result2 = await improved.retrieve(question, chapter_id, semantic_weight=0.5, keyword_weight=0.5, rerank=True, use_context_filter=True, top_k=10, select_k=5)
        print('=== Improved ===')
        for i, chunk in enumerate(result2.chunks[:3], 1):
            print(i, chunk.get('position_index'), chunk['content'][:80].replace('\n',' '))
    finally:
        await pg.disconnect()

import nest_asyncio
nest_asyncio.apply()
await run_inspect()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4366.05it/s]
c:\Users\vansh\OneDrive\Desktop\PrepPanda\Server\Core\Parser\embedder.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._model.get_sentence_embedding_dimension()


=== Baseline ===
1 1 India was amongst the first countries in the world to initiate action plans and 
2 2 Problems and Strategies
3 3 Population Explosion and
---
=== Improved ===
1 18 sexual co-habitation. The reasons for this could be many–physical, congenital, d
2 28 Suggest some methods to assist infertile couples to have children.
3 17 A discussion on reproductive health is incomplete without a mention of infertili


In [17]:
import os
import sys
from pathlib import Path
import asyncio
import nest_asyncio

ROOT = Path.cwd().parent
SERVER = ROOT / 'Server'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SERVER))

from Core.Storage.PostgresHandler import PostgresHandler
from Core.Parser.embedder import ChunkEmbedder
from Core.SRS.retriever import Retriever
from evaluation.retrieval_wrapper import ImprovedRetriever

os.environ.setdefault('POSTGRES_USER', 'preppanda')
os.environ.setdefault('POSTGRES_PASSWORD', 'preppass')
os.environ.setdefault('POSTGRES_DB', 'appdb')
os.environ.setdefault('POSTGRES_HOST', 'localhost')
os.environ.setdefault('POSTGRES_PORT', '5432')

nest_asyncio.apply()

async def inspect_all():
    pg = PostgresHandler()
    await pg.connect()
    try:
        embedder = ChunkEmbedder()
        ret = ImprovedRetriever(pg=pg, embedder=embedder)
        for item in dataset['questions']:
            print('QUESTION:', item['question'])
            print('chapter_id:', item['chapter_id'])
            res = await ret.retrieve(item['question'], item['chapter_id'], semantic_weight=0.5, keyword_weight=0.5, rerank=True, use_context_filter=True, top_k=10, select_k=5)
            for idx, chunk in enumerate(res.chunks[:3], 1):
                text = chunk.get('content','').replace('\n',' ')
                print(f'  {idx}. pos={chunk.get("position_index")} {text[:120]}')
            print()
    finally:
        await pg.disconnect()

await inspect_all()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7021.08it/s]
c:\Users\vansh\OneDrive\Desktop\PrepPanda\Server\Core\Parser\embedder.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._model.get_sentence_embedding_dimension()


QUESTION: What is spermatogenesis?
chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  1. pos=18 sexual co-habitation. The reasons for this could be many–physical, congenital, diseases, drugs, immunological or even ps
  2. pos=28 Suggest some methods to assist infertile couples to have children.
  3. pos=17 A discussion on reproductive health is incomplete without a mention of infertility. A large number of couples all over t

QUESTION: Describe structure of testis
chapter_id: e9b3f002-f293-40fe-8591-87f6dc44da5d
  1. pos=18 sexual co-habitation. The reasons for this could be many–physical, congenital, diseases, drugs, immunological or even ps
  2. pos=16 Infections or diseases which are transmitted through sexual intercourse are collectively called sexually transmitted inf
  3. pos=0 You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a closely related top

QUESTION: What is fertilisation in humans?
chapter_id: e9b3f002-f293-40fe-8591-87f6

In [18]:
import json
import os
import sys
from pathlib import Path
import asyncio
import nest_asyncio

ROOT = Path.cwd().parent
SERVER = ROOT / 'Server'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SERVER))

from Core.Storage.PostgresHandler import PostgresHandler
from Core.Parser.embedder import ChunkEmbedder
from evaluation.retrieval_wrapper import ImprovedRetriever

os.environ.setdefault('POSTGRES_USER', 'preppanda')
os.environ.setdefault('POSTGRES_PASSWORD', 'preppass')
os.environ.setdefault('POSTGRES_DB', 'appdb')
os.environ.setdefault('POSTGRES_HOST', 'localhost')
os.environ.setdefault('POSTGRES_PORT', '5432')

nest_asyncio.apply()

async def collect_top_chunks():
    pg = PostgresHandler()
    await pg.connect()
    try:
        embedder = ChunkEmbedder()
        improved = ImprovedRetriever(pg=pg, embedder=embedder)
        summary = []
        for item in dataset['questions'][:10]:
            res = await improved.retrieve(item['question'], item['chapter_id'], semantic_weight=0.5, keyword_weight=0.5, rerank=True, use_context_filter=True, top_k=10, select_k=5)
            summary.append({
                'question': item['question'],
                'chapter_id': item['chapter_id'],
                'chunks': [
                    {
                        'position': chunk.get('position_index'),
                        'content': chunk.get('content','')[:180].replace('\n',' '),
                    }
                    for chunk in res.chunks[:5]
                ],
            })
        (ROOT / 'evaluation' / 'inspect_retrieval.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    finally:
        await pg.disconnect()

await collect_top_chunks()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9257.82it/s]
c:\Users\vansh\OneDrive\Desktop\PrepPanda\Server\Core\Parser\embedder.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._model.get_sentence_embedding_dimension()


In [19]:
import os
import sys
from pathlib import Path
import asyncio
from pprint import pprint

ROOT = Path.cwd().parent
SERVER = ROOT / 'Server'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SERVER))

from Core.Storage.PostgresHandler import PostgresHandler
from evaluation.dataset_builder import QUESTIONS, select_chapter_id, get_chapter_map

os.environ.setdefault('POSTGRES_USER', 'preppanda')
os.environ.setdefault('POSTGRES_PASSWORD', 'preppass')
os.environ.setdefault('POSTGRES_DB', 'appdb')
os.environ.setdefault('POSTGRES_HOST', 'localhost')
os.environ.setdefault('POSTGRES_PORT', '5432')

async def test_mapping():
    pg = PostgresHandler()
    await pg.connect()
    try:
        chapter_map = await get_chapter_map(pg)
        print('chapter_map keys:', list(chapter_map.keys()))
        for q in QUESTIONS:
            chapter_id = select_chapter_id(q, chapter_map)
            print(q, '=>', chapter_id)
    finally:
        await pg.disconnect()

import nest_asyncio
nest_asyncio.apply()
await test_mapping()


chapter_map keys: ['Sexual Reproduction in Flowering Plants', 'Human Reproduction', 'Reproductive Health', 'Principles of Inheritance and Variation', 'Molecular Basis of Inheritance', 'Evolution', 'Human Health and Disease', 'Strategies for Enhancement in Food Production', 'Microbes in Human Welfare', 'Biotechnology: Principles and Processes', 'Biotechnology and Its Applications', 'Organisms and Populations', 'Reproduction in Organisms']
What is spermatogenesis? => e9b3f002-f293-40fe-8591-87f6dc44da5d
Describe structure of testis => e9b3f002-f293-40fe-8591-87f6dc44da5d
What is fertilisation in humans? => e9b3f002-f293-40fe-8591-87f6dc44da5d
Explain implantation process => e9b3f002-f293-40fe-8591-87f6dc44da5d
What is role of placenta? => e9b3f002-f293-40fe-8591-87f6dc44da5d
What is double fertilisation? => e9b3f002-f293-40fe-8591-87f6dc44da5d
Explain types of pollination => 4450becc-0d5f-4587-9a16-7a17e1b111d8
What is apomixis? => 4450becc-0d5f-4587-9a16-7a17e1b111d8
Describe pre-fertil

In [20]:
import os
import sys
from pathlib import Path
import asyncio
import nest_asyncio

ROOT = Path.cwd().parent
SERVER = ROOT / 'Server'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SERVER))

from evaluation.dataset_builder import build_dataset
from evaluation.labeling import label_dataset

os.environ.setdefault('POSTGRES_USER', 'preppanda')
os.environ.setdefault('POSTGRES_PASSWORD', 'preppass')
os.environ.setdefault('POSTGRES_DB', 'appdb')
os.environ.setdefault('POSTGRES_HOST', 'localhost')
os.environ.setdefault('POSTGRES_PORT', '5432')

nest_asyncio.apply()

print('Rebuilding dataset and labels...')
await build_dataset()
await label_dataset(auto=True)
print('Done')


Rebuilding dataset and labels...
Found 13 Biology chapter(s) for dataset creation.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4427.21it/s]
c:\Users\vansh\OneDrive\Desktop\PrepPanda\Server\Core\Parser\embedder.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._model.get_sentence_embedding_dimension()


Question 'What is spermatogenesis?' → chapter_id=e9b3f002-f293-40fe-8591-87f6dc44da5d
Question 'Describe structure of testis' → chapter_id=e9b3f002-f293-40fe-8591-87f6dc44da5d
Question 'What is fertilisation in humans?' → chapter_id=e9b3f002-f293-40fe-8591-87f6dc44da5d
Question 'Explain implantation process' → chapter_id=e9b3f002-f293-40fe-8591-87f6dc44da5d
Question 'What is role of placenta?' → chapter_id=e9b3f002-f293-40fe-8591-87f6dc44da5d
Question 'What is double fertilisation?' → chapter_id=e9b3f002-f293-40fe-8591-87f6dc44da5d
Question 'Explain types of pollination' → chapter_id=4450becc-0d5f-4587-9a16-7a17e1b111d8
Question 'What is apomixis?' → chapter_id=4450becc-0d5f-4587-9a16-7a17e1b111d8
Question 'Describe pre-fertilisation events' → chapter_id=e9b3f002-f293-40fe-8591-87f6dc44da5d
Question 'What is DNA structure?' → chapter_id=1e04530f-c596-49a3-85e3-b44bc655daed
Question 'What is genetic code?' → chapter_id=1e04530f-c596-49a3-85e3-b44bc655daed
Question 'Explain transcription

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9236.00it/s]


Labeled dataset written to C:\Users\vansh\OneDrive\Desktop\PrepPanda\evaluation\dataset_labeled.json
Dataset is marked as pseudo_ground_truth.


TypeError: object NoneType can't be used in 'await' expression

In [25]:
import importlib
import evaluation.dataset_builder as db
importlib.reload(db)

print('normalize:', db.normalize_text('What is double fertilisation?'))
for q in db.QUESTIONS:
    chapter_id = db.select_chapter_id(q, {
        'Human Reproduction': 'A',
        'Sexual Reproduction in Flowering Plants': 'B',
        'Molecular Basis of Inheritance': 'C',
        'Reproductive Health': 'D',
        'Principles of Inheritance and Variation': 'E',
        'Reproduction in Organisms': 'F',
    })
    print(q, '=>', chapter_id)


normalize: what is double fertilisation
What is spermatogenesis? => A
Describe structure of testis => A
What is fertilisation in humans? => A
Explain implantation process => A
What is role of placenta? => A
What is double fertilisation? => B
Explain types of pollination => B
What is apomixis? => B
Describe pre-fertilisation events => A
What is DNA structure? => C
What is genetic code? => C
Explain transcription => C
What is reproductive health? => D
What are family planning methods? => D
What are Mendel’s laws? => E


In [24]:
import os
import asyncio
import nest_asyncio
from Core.Storage.PostgresHandler import PostgresHandler

os.environ.setdefault('POSTGRES_USER', 'preppanda')
os.environ.setdefault('POSTGRES_PASSWORD', 'preppass')
os.environ.setdefault('POSTGRES_DB', 'appdb')
os.environ.setdefault('POSTGRES_HOST', 'localhost')
os.environ.setdefault('POSTGRES_PORT', '5432')

nest_asyncio.apply()

async def get_map():
    pg = PostgresHandler()
    await pg.connect()
    try:
        return await db.get_chapter_map(pg)
    finally:
        await pg.disconnect()

chapter_map = asyncio.get_event_loop().run_until_complete(get_map())
for q in db.QUESTIONS:
    chapter_id = db.select_chapter_id(q, chapter_map)
    print(q, '=>', chapter_id)


What is spermatogenesis? => e9b3f002-f293-40fe-8591-87f6dc44da5d
Describe structure of testis => e9b3f002-f293-40fe-8591-87f6dc44da5d
What is fertilisation in humans? => e9b3f002-f293-40fe-8591-87f6dc44da5d
Explain implantation process => e9b3f002-f293-40fe-8591-87f6dc44da5d
What is role of placenta? => e9b3f002-f293-40fe-8591-87f6dc44da5d
What is double fertilisation? => 4450becc-0d5f-4587-9a16-7a17e1b111d8
Explain types of pollination => 4450becc-0d5f-4587-9a16-7a17e1b111d8
What is apomixis? => 4450becc-0d5f-4587-9a16-7a17e1b111d8
Describe pre-fertilisation events => e9b3f002-f293-40fe-8591-87f6dc44da5d
What is DNA structure? => 1e04530f-c596-49a3-85e3-b44bc655daed
What is genetic code? => 1e04530f-c596-49a3-85e3-b44bc655daed
Explain transcription => 1e04530f-c596-49a3-85e3-b44bc655daed
What is reproductive health? => 7ba50bb0-0e79-43dd-9525-b5fae00149d9
What are family planning methods? => 7ba50bb0-0e79-43dd-9525-b5fae00149d9
What are Mendel’s laws? => fba40315-57e4-4ca9-b72b-e69b15

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')
question = 'What is spermatogenesis?'
texts = [chunk['text'] for chunk in dataset['questions'][0]['chunks']]
emb_q = model.encode([question], normalize_embeddings=True)[0]
emb_texts = model.encode(texts, normalize_embeddings=True)
for idx, (text, emb) in enumerate(zip(texts, emb_texts), 1):
    print(idx, float(np.dot(emb_q, emb)), text[:120].replace('\n', ' '))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4673.20it/s]


1 0.24043643474578857 India was amongst the first countries in the world to initiate action plans and programmes at a national level to attain
2 0.09341637790203094 Problems and Strategies
3 0.22092823684215546 Population Explosion and
4 0.3337825536727905 reproduction-related areas are currently in operation under the popular name ‘Reproductive and Child Health Care (RCH) p
5 0.287306010723114 by scientists at Central Drug Research Institute (CDRI) in Lucknow, India? Better awareness about sex related matters, i


In [16]:
import os
import asyncio
import nest_asyncio
import asyncpg
import numpy as np
from sentence_transformers import SentenceTransformer

nest_asyncio.apply()

dsn = (
    os.environ.get('DATABASE_URL') or
    f"postgresql://{os.environ.get('POSTGRES_USER','preppanda')}:{os.environ.get('POSTGRES_PASSWORD','preppass')}@"
    f"{os.environ.get('POSTGRES_HOST','localhost')}:{os.environ.get('POSTGRES_PORT','5432')}/"
    f"{os.environ.get('POSTGRES_DB','appdb')}"
)

async def calc_sim(chapter_id, question, limit=100):
    conn = await asyncpg.connect(dsn=dsn)
    try:
        rows = await conn.fetch(
            '''SELECT position_index, content FROM core.chunks WHERE chapter_id = $1 ORDER BY position_index LIMIT $2''',
            chapter_id,
            limit,
        )
        texts = [r['content'] for r in rows]
        model = SentenceTransformer('all-mpnet-base-v2')
        q_emb = model.encode([question], normalize_embeddings=True)[0]
        emb = model.encode(texts, normalize_embeddings=True)
        sims = [(r['position_index'], float(np.dot(q_emb, e)), r['content'][:120].replace('\n',' ')) for r, e in zip(rows, emb)]
        sims.sort(key=lambda x: x[1], reverse=True)
        for pos, score, text in sims[:10]:
            print(pos, score, text)
    finally:
        await conn.close()

await calc_sim('e9b3f002-f293-40fe-8591-87f6dc44da5d', 'What is spermatogenesis?', limit=100)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6645.22it/s]


18 0.5137909054756165 sexual co-habitation. The reasons for this could be many–physical, congenital, diseases, drugs, immunological or even ps
28 0.43219107389450073 Suggest some methods to assist infertile couples to have children.
20 0.40130653977394104 What do you think is the significance of reproductive health in a society?
10 0.3914591372013092 Natural methods work on the principle of avoiding chances of ovum and sperms meeting. Periodic abstinence is one such me
12 0.38306882977485657 pregnancies. Surgical intervention blocks gamete transport and thereby prevent conception. Sterilisation procedure in th
0 0.3802671432495117 You have learnt about human reproductive system and its functions in Chapter 2. Now, let’s discuss a closely related top
21 0.37725257873535156 Suggest the aspects of reproductive health which need to be given special attention in the present scenario.
26 0.368880033493042 Removal of gonads cannot be considered as a contraceptive option.  Why?
17 0.3666838407